In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [3]:
class BatsmanState(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes: int
    sr: float
    bbp: float
    boundary_percent: float
    summary: str

In [ ]:
def calculate_sr(state: BatsmanState) -> BatsmanState:
    state['sr'] = (state['runs'] / state['balls']) * 100
    return state

def calculate_bbp(state: BatsmanState) -> BatsmanState:
    bp = state['balls'] / (state['fours'] + state['sixes'])
    state['bbp'] = bp
    return state

def calculate_boundary_percent(state: BatsmanState) -> BatsmanState:
    total_boundaries = state['fours'] * 4 + state['sixes'] * 6
    state['boundary_percent'] = total_boundaries / state['runs'] * 100 if total_boundaries > 0 else 0
    return state

def summary(state: BatsmanState) -> BatsmanState:
    summary = (f"Runs: {state['runs']}, Balls: {state['balls']}, Fours: {state['fours']}, "
               f"Sixes: {state['sixes']}, Strike Rate: {state['sr']:.2f}, Boundary Percentage: {state['boundary_percent']:.2f}%")
    state['summary'] = summary
    return state

In [5]:
graph = StateGraph(BatsmanState)
graph.add_node('calculate_sr', calculate_sr)
graph.add_node('calculate_bbp', calculate_bbp)
graph.add_node('calculate_boundary_percent', calculate_boundary_percent)
graph.add_node('summary', summary)

In [6]:
# adding edges to define the workflow
graph.add_edge(START, 'calculate_sr')
graph.add_edge(START, 'calculate_bbp')
graph.add_edge(START, 'calculate_boundary_percent')
graph.add_edge('calculate_sr', 'summary')
graph.add_edge('calculate_bbp', 'summary')
graph.add_edge('calculate_boundary_percent', 'summary')
graph.add_edge('summary', END)

In [8]:
wf = graph.compile()

In [10]:
initial_state = {
    'runs': 120,
    'balls': 80,
    'fours': 10,
    'sixes': 5
}
final_state = wf.invoke(initial_state)
print(final_state['summary'])

InvalidUpdateError: At key 'runs': Can receive only one value per step. Use an Annotated key to handle multiple values.
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/INVALID_CONCURRENT_GRAPH_UPDATE